In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
llm=ChatOllama(model="llama3.2",
               temperature=0)

In [ ]:
class JokeSchema(TypedDict):
    topic:str
    joke:str
    explanation:str

In [ ]:
def generate_joke(state:JokeSchema)->JokeSchema:
    prompt=f'Write a joke on the topic {state['topic']}'
    response=llm.invoke(prompt)
    return {'joke':response}
    

In [ ]:
def explain_joke(state:JokeSchema)->JokeSchema:
    prompt=f'Write an explanation of the joke {state['joke']}'
    response=llm.invoke(prompt)
    return {'explanation':response}

In [ ]:
checkpointer=InMemorySaver()
graph=StateGraph(JokeSchema)
graph.add_node('generate_joke',generate_joke)
graph.add_node('explain_joke',explain_joke)
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','explain_joke')
graph.add_edge('explain_joke',END)
workflow=graph.compile(checkpointer=checkpointer)
graph.compile()

In [ ]:
config={'configurable':{'thread_id':"1"}}
output=workflow.invoke({'topic':'pizza'},config=config)


In [ ]:
output['joke'].content

In [ ]:
workflow.get_state((config))

In [ ]:
list(workflow.get_state_history((config)))